# L-CAD baseline on the Phase 0 smoke subset (Colab)

L-CAD (NeurIPS 2023, language-based colorization with a diffusion prior) is the **language baseline** for chroma-reasoner Phase 0. It does not fit in 4 GB VRAM, so it runs here on a Colab GPU (T4/A100 both fine).

Flow: clone repos → install deps → download weights (Google Drive) → regenerate the deterministic COCO smoke subset → run inference with COCO captions → zip outputs for local evaluation with `scripts/evaluate.py`.

> **Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Clone repos

In [ ]:
# TODO: replace with your GitHub URL once chroma-reasoner is pushed.
CHROMA_REASONER_URL = "https://github.com/<YOUR_USERNAME>/chroma-reasoner.git"

!git clone https://github.com/changzheng123/L-CAD.git
!git clone {CHROMA_REASONER_URL}

## 2. Install dependencies

In [ ]:
%cd /content
# Colab runtimes are ephemeral: re-run the clone cell after any reset.
# These guards make this cell safe to re-run either way.
!test -d L-CAD || git clone https://github.com/changzheng123/L-CAD.git

# Do NOT `pip install -r L-CAD/requirements.txt`: it is a conda-env dump that
# pins torch==1.12.1, numpy from a local file:// path, and an editable local
# segment-anything — it breaks on Colab and would clobber Colab's CUDA torch.
# Install only what the LDM-style codebase imports, keeping Colab's torch.
!pip install -q pytorch-lightning==1.9.5 omegaconf einops kornia open-clip-torch taming-transformers-rom1504
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gdown pycocotools clean-fid
!pip install -q -e chroma-reasoner

## 3. Download pretrained weights

Weights folder (from the L-CAD README): https://drive.google.com/drive/folders/1_zVJrp_UkFDaZpcC8aLzpv4iPsHADQU-

In [ ]:
!mkdir -p L-CAD/models
!gdown --folder https://drive.google.com/drive/folders/1_zVJrp_UkFDaZpcC8aLzpv4iPsHADQU- -O L-CAD/models --remaining-ok
!ls -lh L-CAD/models

## 4. Regenerate the smoke subset (deterministic: n=300, seed=0)

Same selection code as local, so stems match for evaluation.

In [ ]:
!cd chroma-reasoner && python scripts/download_coco_subset.py --n 300 --seed 0 --root data/coco

## 5. Run L-CAD inference

Resolved mechanism (from reading the L-CAD source): `python colorization_main.py -m`
(multicolor, no SAM) loads images from `L-CAD/example/` and captions from
`L-CAD/example/test-pair.json`, a **list of `[filename, caption]` pairs**
(`MyDataset(img_dir='example', caption_dir='example', split='test')`).
The checkpoint path is hardcoded as a placeholder (`.models/xxxxx.ckpt`)
in `colorization_main.py` and must be patched — the cell below does it
with `sed`. Outputs land in `L-CAD/image_log/test_{timestamp}_ug_4.5/`
(DDIM 50 steps, guidance 4.5) with the prompt appended to each filename;
the collect cell renames them to `{stem}.png` for evaluation.

In [ ]:
import json, pathlib
manifest = json.load(open('chroma-reasoner/data/coco/manifest.json'))
pairs = [(im['file_name'], im['captions'][0]) for im in manifest['images'] if im['captions']]
print(len(pairs), 'image/caption pairs; first:', pairs[0])

# Write a simple annotation file many L-CAD configs can be adapted to consume.
with open('lcad_smoke_captions.json', 'w') as f:
    json.dump({fn: cap for fn, cap in pairs}, f, indent=2)

In [ ]:
import json, glob, os, shutil

# Stage inputs the way L-CAD's demo branch expects:
# example/<file_name> + example/test-pair.json = list of [filename, caption].
# Feeding color images is leak-free: the model only consumes the L channel.
src = 'chroma-reasoner/data/coco/val2017_subset'
for fn, _ in pairs:
    shutil.copy(os.path.join(src, fn), os.path.join('L-CAD/example', fn))
with open('L-CAD/example/test-pair.json', 'w') as f:
    json.dump(pairs, f, indent=2)

# colorization_main.py hardcodes resume_path='.models/xxxxx.ckpt' (a placeholder).
# Point it at the largest downloaded .ckpt (the colorization model; the small one is the VAE).
ckpts = glob.glob('L-CAD/models/**/*.ckpt', recursive=True)
print('checkpoints found:', [(c, f'{os.path.getsize(c)/1e9:.1f} GB') for c in ckpts])
rel = os.path.relpath(max(ckpts, key=os.path.getsize), 'L-CAD')
!sed -i "s|resume_path *= *'[^']*'|resume_path='{rel}'|" L-CAD/colorization_main.py
!grep -n "resume_path" L-CAD/colorization_main.py

In [ ]:
# 50-step DDIM x 300 images: roughly 2-4 h on T4, ~40 min on A100.
# For a quick first pass, rewrite test-pair.json with pairs[:20] above and re-run.
!cd L-CAD && python colorization_main.py -m

In [ ]:
import glob, os, re, shutil

# L-CAD saves to L-CAD/image_log/test_{timestamp}_ug_4.5/ and appends the
# prompt to filenames; rename back to {stem}.png so evaluate.py can pair
# outputs with ground truth. COCO stems are all digits.
runs = sorted(glob.glob('L-CAD/image_log/test_*'), key=os.path.getmtime)
out_dir = 'results/lcad'; os.makedirs(out_dir, exist_ok=True)
n = 0
for p in glob.glob(os.path.join(runs[-1], '*.png')):
    stem = re.match(r'\d+', os.path.basename(p)).group(0)
    shutil.copy(p, os.path.join(out_dir, stem + '.png')); n += 1
print(n, 'outputs ->', out_dir)

## 6. Export outputs for local evaluation

In [ ]:
!zip -rq lcad_outputs.zip results/lcad
from google.colab import files
files.download('lcad_outputs.zip')
# Locally: python scripts/evaluate.py --pred results/lcad --gt data/coco/val2017_subset \
#              --manifest data/coco/manifest.json --out results/phase0/lcad